In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from amms.core import products
from amms.core.analysis.sfr import lookback_edges, sfh
from amms.core.datasets import b12, patel2020
from amms.core.analysis import units

DATASET, R_MAX, DT = "ext_b12-model2", 5.0, 0.1 # kpc, Gyr

In [2]:
def load_sfh(galaxy, snap=69, r_max=R_MAX, dt=DT):
    p = products.read_product(products.product_path(DATASET, "newstars", f"{galaxy}-{snap:03d}"))
    t_now = p.attrs.meta["t_now_gyr"]
    edges = lookback_edges(t_now, dt)
    pos, t, m = p.arrays["pos_kpc"], p.arrays["t_form_gyr"], p.arrays["mass_msun"]
    sel = np.linalg.norm(pos, axis=1) < r_max # position is already in the galaxy's own frame
    _, rate = sfh(t[sel], m[sel], bins=edges)
    _, rate_all = sfh(t, m, bins=edges)
    return p, t_now, edges, rate, rate_all

In [3]:
def load_orbits(model):
    ds = f"ext_b12-model{model}"
    o = {g: products.read_product(products.product_path(ds, "orbit", g)).arrays for g in ("lmc", "smc")}
    assert np.array_equal(o["lmc"]["snapshot"], o["smc"]["snapshot"])
    # present day taken from the same header that the newstars product used
    # SFH and orbit panels then share t_now
    t_now = float(o["lmc"]["t_gyr"][o["lmc"]["snapshot"] == b12.SNAP_PRESENT_DAY][0])
    return o, t_now

In [4]:
p, t_now, edges, rate, rate_all = load_sfh("lmc")
print(p.attrs.meta["gadget_time_to_gyr"], "t_now", t_now, "dropped ft==0", p.attrs.meta["n_dropped_ft0"])
print("PIN all LMC stars, last 100 Myr:", rate_all[-1], "(0.0488 with the old 0.97)")

KeyError: 'local'

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
# reversed so stairs get increasing lookback edges with 0 as the present day
ax.stairs(rate[::-1], (t_now - edges)[::-1], label=f"r < {R_MAX:g} kpc")
ax.stairs(rate_all[::-1], (t_now - edges)[::-1], ls="--", label="all LMC")
ax.set(xlabel="lookback time [Gyr]", ylabel=r"SFR [$M_\odot\,\mathrm{yr}^{-1}$]")
ax.legend()

# Orbits

In [ ]:
for model in (2, 1):
    o, t_now = load_orbits(model)
    for g, a in o.items():
        v_fd = np.linalg.norm(np.diff(a["pos_kpc"], axis=0), axis=1) / np.diff(a["t_gyr"]) / units.KPC_PER_GYR_PER_KMS
        speed = np.linalg.norm(a["vel_kms"], axis=1)
        ratio = v_fd / (0.5 * (speed[1:] + speed[:-1]))
        worst = int(np.argmax(np.abs(ratio - 1)))
        print(f"M{model} {g}: ratio {ratio.min():.3f}-{ratio.max():.3f}, worst at snap {a['snapshot'][worst + 1]}")

In [ ]:
COLOR= {1: "C0", 2: "C3"}
MODELS = (2,)

fig, ax = plt.subplots(figsize=(6, 3.5))
for model in MODELS:
    o, t_now = load_orbits(model)
    lmc, smc = o["lmc"], o["smc"]
    lb = t_now - lmc["t_gyr"]
    keep = lb >= -1e-9 # only keep up to present day
    c = COLOR[model]
    ax.plot(lb[keep], np.linalg.norm(lmc["pos_kpc"] - smc["pos_kpc"], axis=1)[keep], c=c, label=f"M{model} LMC-SMC")
    ax.plot(lb[keep], np.linalg.norm(lmc["pos_kpc"], axis=1)[keep], c=c, ls="--", label=f"M{model} LMC-MW")
    ax.plot(lb[keep], np.linalg.norm(smc["pos_kpc"], axis=1)[keep], c=c, ls=":", label=f"M{model} SMC-MW")

# Add in the present day location from Patel+2020 at lookback=0
L, S = np.asarray(patel2020.LMC_CENTER), np.asarray(patel2020.SMC_CENTER)
for d, mk in ((np.linalg.norm(L - S), "o"), (np.linalg.norm(L), "s"), (np.linalg.norm(S), "^")):
    ax.plot(0, d, mk, c="k", ms=5)
ax.set(xlabel="lookback time [Gyr]", ylabel="distance [kpc]", xlim=(0, None))
ax.legend(fontsize=7)
ax.invert_xaxis()